In [1]:
from google.colab import drive
drive.mount('/content/drive')

# Create a directory for this lab if it doesn't exist
import os
output_path = '/content/drive/MyDrive/CSET419_Lab11'
os.makedirs(output_path, exist_ok=True)
print(f'Outputs will be saved to: {output_path}')

Mounted at /content/drive
Outputs will be saved to: /content/drive/MyDrive/CSET419_Lab11


In [7]:
model_name = "gpt2"
tokenizer = GPT2Tokenizer.from_pretrained(model_name)
model = GPT2LMHeadModel.from_pretrained(model_name)

# GPT-2 does not have a padding token by default, so we add one
tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = model.config.eos_token_id

print(f"Model '{model_name}' loaded and tokenizer configured.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Model 'gpt2' loaded and tokenizer configured.


In [8]:
from datasets import Dataset

# Sample data for Product Review Generator as per typical lab requirements
reviews_data = [
    {"text": "Product: Smartphone | Review: This phone has an amazing camera and long battery life. Highly recommend!"},
    {"text": "Product: Laptop | Review: Great performance for gaming, but the fans can get a bit loud under load."},
    {"text": "Product: Headphones | Review: Excellent sound quality and very comfortable for long listening sessions."},
    {"text": "Product: Smartwatch | Review: The fitness tracking is accurate, but the screen is a bit small for my liking."}
]

def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=128)

# Create dataset
review_dataset = Dataset.from_list(reviews_data)
tokenized_reviews = review_dataset.map(tokenize_function, batched=True)

print("Product review dataset prepared and tokenized.")

Map:   0%|          | 0/4 [00:00<?, ? examples/s]

Product review dataset prepared and tokenized.


In [6]:
!pip install transformers datasets torch
import torch
from transformers import GPT2Tokenizer, GPT2LMHeadModel, Trainer, TrainingArguments

print("Libraries installed. GPU available:", torch.cuda.is_available())

Libraries installed. GPU available: True


In [14]:
!pip install transformers datasets accelerate -q

import torch, math
from transformers import (GPT2LMHeadModel, GPT2Tokenizer, Trainer,
    TrainingArguments, DataCollatorForLanguageModeling, set_seed)
from datasets import Dataset
import os

set_seed(42)

model_name = 'gpt2'
tokenizer = GPT2Tokenizer.from_pretrained(model_name)
model = GPT2LMHeadModel.from_pretrained(model_name)

tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.eos_token_id

print("Model and tokenizer re-initialized for Component-I.")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model and tokenizer re-initialized for Component-I.


In [15]:
def generate_text(model, tokenizer, prompt, max_length=100):
    model.eval()
    inputs = tokenizer.encode(prompt, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(inputs, max_length=max_length, temperature=0.8,
            top_k=50, do_sample=True, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0], skip_special_tokens=True)

review_prompts = [
    'This product is',
    'I bought this phone and',
    'The quality of this item',
]

print('=== BASELINE REVIEWS (Before Fine-Tuning) ===')
baseline = {}
for p in review_prompts:
    baseline[p] = generate_text(model, tokenizer, p)
    print(f'Prompt: {p}\nOutput: {baseline[p]}\n')

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


=== BASELINE REVIEWS (Before Fine-Tuning) ===
Prompt: This product is
Output: This product is made from high quality, lightweight stainless steel. If you are looking for something a little more durable, it's a good choice.

Laser Pouch

Not all of our laser printers are created equal. We have a laser printer that comes with all of our printer parts. These parts include our new 3D printer and a 3D printed printing service. All of our printers make laser printers, including our laser printers, using laser technology. Our laser printers are the most

Prompt: I bought this phone and
Output: I bought this phone and I have not used it on a lot of people. I have also not used it on any other people.

The screen was amazing and the sound was amazing. It was not loud. I would never use it on a tv, laptop, smartphone or other connected device in the future.

The battery life is good. The phone works great but it has so many problems.

I have been using phones that have the Snapdragon 616 process

In [21]:
# Step 1: Setup, Load Model and Drive
from google.colab import drive
import torch, math, os
from transformers import (GPT2LMHeadModel, GPT2Tokenizer, Trainer,
    TrainingArguments, DataCollatorForLanguageModeling, set_seed)
from datasets import Dataset

drive.mount('/content/drive')
output_path = '/content/drive/MyDrive/CSET419_Lab11'
os.makedirs(output_path, exist_ok=True)

set_seed(42)
model_name = 'gpt2'
tokenizer = GPT2Tokenizer.from_pretrained(model_name)
model = GPT2LMHeadModel.from_pretrained(model_name)

tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.eos_token_id

print(f"Setup complete. Outputs will be saved to: {output_path}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Setup complete. Outputs will be saved to: /content/drive/MyDrive/CSET419_Lab11


In [22]:
# Component-I: Step 2 - Baseline Reviews
def generate_text(model, tokenizer, prompt, max_length=100):
    model.eval()
    inputs = tokenizer.encode(prompt, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(inputs, max_length=max_length, temperature=0.8,
            top_k=50, do_sample=True, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0], skip_special_tokens=True)

review_prompts = ['This product is', 'I bought this phone and', 'The quality of this item']

print('=== BASELINE REVIEWS (Before Fine-Tuning) ===')
baseline = {p: generate_text(model, tokenizer, p) for p in review_prompts}
for p, output in baseline.items():
    print(f'Prompt: {p}\nOutput: {output}\n')

=== BASELINE REVIEWS (Before Fine-Tuning) ===
Prompt: This product is
Output: This product is made from high quality, lightweight stainless steel. If you are looking for something a little more durable, it's a good choice.

Laser Pouch

Not all of our laser printers are created equal. We have a laser printer that comes with all of our printer parts. These parts include our new 3D printer and a 3D printed printing service. All of our printers make laser printers, including our laser printers, using laser technology. Our laser printers are the most

Prompt: I bought this phone and
Output: I bought this phone and I have not used it on a lot of people. I have also not used it on any other people.

The screen was amazing and the sound was amazing. It was not loud. I would never use it on a tv, laptop, smartphone or other connected device in the future.

The battery life is good. The phone works great but it has so many problems.

I have been using phones that have the Snapdragon 616 process

In [23]:
# Component-I: Step 3 - Prepare Dataset and Fine-Tune
corpus = [
    'this phone has an amazing battery life and the camera quality is outstanding for the price.',
    'i bought this laptop for college and it handles all my assignments and coding projects perfectly.',
    'the sound quality of these headphones is incredible with deep bass and clear vocals.',
    'this smartwatch tracks my steps accurately and the heart rate monitor is very reliable.',
    'great wireless earbuds with noise cancellation that blocks out all background sound.',
    'the keyboard feels very comfortable for long typing sessions and the backlight is a nice touch.',
    'this portable charger saved me during travel and it charges my phone three times on a single charge.',
    'the tablet screen is bright and colorful which makes watching movies a great experience.',
    'i love this fitness tracker because it motivates me to reach my daily exercise goals.',
    'this bluetooth speaker is compact but delivers surprisingly loud and clear audio.',
    'the delivery was fast and the product was packed securely with no damage at all.',
    'excellent value for money and the build quality feels premium despite the affordable price.',
    'the customer service team was very helpful when i had questions about the product features.',
    'this camera takes stunning photos in low light and the video recording quality is very smooth.',
    'i have been using this product for three months and it still works perfectly like day one.',
    'the design is sleek and modern and it looks great on my desk next to my other gadgets.',
    'easy to set up right out of the box and the instructions were clear and simple to follow.',
    'highly recommend this product to anyone looking for quality and reliability at a fair price.',
    'the software updates keep adding new features which makes this purchase even more worthwhile.',
    'best purchase i made this year and i would definitely buy from this brand again.',
]

dataset = Dataset.from_dict({'text': corpus})
tokenized = dataset.map(lambda x: tokenizer(x['text'], truncation=True, max_length=128, padding='max_length'), batched=True)
tokenized = tokenized.map(lambda x: {'labels': x['input_ids']}, batched=True)
split = tokenized.train_test_split(test_size=0.15, seed=42)

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
training_args = TrainingArguments(
    output_dir='./gpt2-reviews', num_train_epochs=15, per_device_train_batch_size=4,
    learning_rate=5e-5, weight_decay=0.01, warmup_steps=50, eval_strategy='epoch',
    logging_steps=10, save_strategy='no', fp16=torch.cuda.is_available(), report_to='none'
)

trainer = Trainer(model=model, args=training_args, train_dataset=split['train'], eval_dataset=split['test'], data_collator=data_collator)
trainer.train()

# Save Component 1
model.save_pretrained(os.path.join(output_path, 'component1_product_reviews'))

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss
1,No log,3.348063
2,4.057265,3.247350
3,4.057265,3.106139
4,3.319935,2.981021
5,3.319935,2.881039
6,2.706133,2.803390
7,2.706133,2.756230
8,1.918991,2.740822
9,1.918991,2.765902
10,1.234805,2.867215


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [24]:
# Component-I: Step 4 - Generate and Compare
eval_res = trainer.evaluate()
print(f'Perplexity: {math.exp(eval_res["eval_loss"]):.2f}')

print('\n=== FINE-TUNED REVIEWS (After Fine-Tuning) ===')
for p in review_prompts:
    ft_out = generate_text(model, tokenizer, p)
    print(f'Prompt: {p}\n  Baseline: {baseline[p][:100]}...\n  Fine-Tuned: {ft_out[:100]}\n')

Perplexity: 24.74

=== FINE-TUNED REVIEWS (After Fine-Tuning) ===
Prompt: This product is
  Baseline: This product is made from high quality, lightweight stainless steel. If you are looking for somethin...
  Fine-Tuned: This product is packed with features that make this purchase even more worthwhile. The quality of th

Prompt: I bought this phone and
  Baseline: I bought this phone and I have not used it on a lot of people. I have also not used it on any other ...
  Fine-Tuned: I bought this phone and it handles all my daily chores perfectly. I would definitely buy from this b

Prompt: The quality of this item
  Baseline: The quality of this item in the item description (and if the item is already in stock) will determin...
  Fine-Tuned: The quality of this item is exemplary with very little distortion and no audible hum. The build qual



In [25]:
# Component-II: Recipe Instruction Generator
recipes_corpus = [
    {"text": "Dish: Scrambled Eggs | Instructions: Whisk eggs in a bowl, add salt and pepper. Heat butter in a pan and cook until soft and fluffy."},
    {"text": "Dish: Pasta Marinara | Instructions: Boil water and cook pasta. In another pan, heat olive oil and garlic, add tomato sauce, then mix with pasta."},
    {"text": "Dish: Grilled Cheese | Instructions: Butter two slices of bread. Place cheese between them and grill on a skillet until golden brown on both sides."},
    {"text": "Dish: Fruit Salad | Instructions: Chop apples, bananas, and grapes. Toss them in a bowl with a splash of lime juice and honey."},
    {"text": "Dish: Guacamole | Instructions: Mash ripe avocados in a bowl. Stir in chopped onions, cilantro, lime juice, and a pinch of salt."}
]

recipe_ds = Dataset.from_list(recipes_corpus).map(lambda x: tokenizer(x['text'], truncation=True, padding='max_length', max_length=128), batched=True)
recipe_ds = recipe_ds.map(lambda x: {'labels': x['input_ids']}, batched=True)

recipe_args = TrainingArguments(
    output_dir='./gpt2-recipes', num_train_epochs=10, per_device_train_batch_size=2,
    learning_rate=5e-5, save_strategy='no', report_to='none', fp16=torch.cuda.is_available()
)

recipe_trainer = Trainer(model=model, args=recipe_args, train_dataset=recipe_ds, data_collator=data_collator)
recipe_trainer.train()

model.save_pretrained(os.path.join(output_path, 'component2_recipe_generator'))
print('\n=== TESTING RECIPE GENERATOR ===')
print(generate_text(model, tokenizer, "Dish: Vegetable Stir-fry | Instructions:", max_length=60))

Map:   0%|          | 0/5 [00:00<?, ? examples/s]

Map:   0%|          | 0/5 [00:00<?, ? examples/s]

Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


=== TESTING RECIPE GENERATOR ===
Dish: Vegetable Stir-fry | Instructions: Heat oil in pan and add onion, garlic, and lime juice. Cook until smooth and tender, about 5 minutes on each side. Place on a baking sheet lined with parchment paper. Drizzle with olive oil and sprinkle with salt and pepper


In [18]:
# Component-II: Recipe Instruction Generator

# 1. Prepare Recipe Dataset
recipes_corpus = [
    {"text": "Dish: Scrambled Eggs | Instructions: Whisk eggs in a bowl, add salt and pepper. Heat butter in a pan and cook until soft and fluffy."},
    {"text": "Dish: Pasta Marinara | Instructions: Boil water and cook pasta. In another pan, heat olive oil and garlic, add tomato sauce, then mix with pasta."},
    {"text": "Dish: Grilled Cheese | Instructions: Butter two slices of bread. Place cheese between them and grill on a skillet until golden brown on both sides."},
    {"text": "Dish: Fruit Salad | Instructions: Chop apples, bananas, and grapes. Toss them in a bowl with a splash of lime juice and honey."},
    {"text": "Dish: Guacamole | Instructions: Mash ripe avocados in a bowl. Stir in chopped onions, cilantro, lime juice, and a pinch of salt."}
]

recipe_dataset = Dataset.from_list(recipes_corpus)
tokenized_recipes = recipe_dataset.map(lambda x: tokenizer(x['text'], truncation=True,
    padding='max_length', max_length=128), batched=True)
tokenized_recipes = tokenized_recipes.map(lambda x: {'labels': x['input_ids']}, batched=True)

# 2. Fine-tune for Recipes
recipe_training_args = TrainingArguments(
    output_dir='./gpt2-recipes',
    num_train_epochs=10,
    per_device_train_batch_size=2,
    learning_rate=5e-5,
    save_strategy='no',
    report_to='none',
    fp16=torch.cuda.is_available()
)

recipe_trainer = Trainer(
    model=model, # Continuing from the current state or re-loading base gpt2
    args=recipe_training_args,
    train_dataset=tokenized_recipes,
    data_collator=data_collator
)

print("Starting fine-tuning for Recipe Generator...")
recipe_trainer.train()

# 3. Save Recipe Model
recipe_save_path = '/content/drive/MyDrive/CSET419_Lab11/component2_recipe_generator'
model.save_pretrained(recipe_save_path)
tokenizer.save_pretrained(recipe_save_path)
print(f"Recipe model saved to {recipe_save_path}")

# 4. Final Verification
print("\n=== TESTING RECIPE GENERATOR ===")
recipe_prompt = "Dish: Vegetable Stir-fry | Instructions:"
print(generate_text(model, tokenizer, recipe_prompt, max_length=60))

Map:   0%|          | 0/5 [00:00<?, ? examples/s]

Map:   0%|          | 0/5 [00:00<?, ? examples/s]

Starting fine-tuning for Recipe Generator...


Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Recipe model saved to /content/drive/MyDrive/CSET419_Lab11/component2_recipe_generator

=== TESTING RECIPE GENERATOR ===
Dish: Vegetable Stir-fry | Instructions: Heat oil in pan and add onion, garlic, and lime juice. Cook until smooth and tender, about 5 minutes on each side. Place on a baking sheet lined with parchment paper. Drizzle with olive oil and sprinkle with salt and pepper


In [19]:
# Component-II: Recipe Instruction Generator

# 1. Prepare Recipe Dataset
recipes_corpus = [
    {"text": "Dish: Scrambled Eggs | Instructions: Whisk eggs in a bowl, add salt and pepper. Heat butter in a pan and cook until soft and fluffy."},
    {"text": "Dish: Pasta Marinara | Instructions: Boil water and cook pasta. In another pan, heat olive oil and garlic, add tomato sauce, then mix with pasta."},
    {"text": "Dish: Grilled Cheese | Instructions: Butter two slices of bread. Place cheese between them and grill on a skillet until golden brown on both sides."},
    {"text": "Dish: Fruit Salad | Instructions: Chop apples, bananas, and grapes. Toss them in a bowl with a splash of lime juice and honey."},
    {"text": "Dish: Guacamole | Instructions: Mash ripe avocados in a bowl. Stir in chopped onions, cilantro, lime juice, and a pinch of salt."}
]

from datasets import Dataset
recipe_dataset = Dataset.from_list(recipes_corpus)
tokenized_recipes = recipe_dataset.map(lambda x: tokenizer(x['text'], truncation=True,
    padding='max_length', max_length=128), batched=True)
tokenized_recipes = tokenized_recipes.map(lambda x: {'labels': x['input_ids']}, batched=True)

# 2. Fine-tune for Recipes
recipe_training_args = TrainingArguments(
    output_dir='./gpt2-recipes',
    num_train_epochs=10,
    per_device_train_batch_size=2,
    learning_rate=5e-5,
    save_strategy='no',
    report_to='none',
    fp16=torch.cuda.is_available()
)

recipe_trainer = Trainer(
    model=model,
    args=recipe_training_args,
    train_dataset=tokenized_recipes,
    data_collator=data_collator
)

print("Starting fine-tuning for Recipe Generator...")
recipe_trainer.train()

# 3. Save Recipe Model to Drive
recipe_save_path = '/content/drive/MyDrive/CSET419_Lab11/component2_recipe_generator'
model.save_pretrained(recipe_save_path)
tokenizer.save_pretrained(recipe_save_path)
print(f"\nRecipe model saved to {recipe_save_path}")

# 4. Final Verification
print("\n=== TESTING RECIPE GENERATOR ===")
recipe_prompt = "Dish: Vegetable Stir-fry | Instructions:"
# Using the generate_text function defined in Step 2
print(generate_text(model, tokenizer, recipe_prompt, max_length=60))

Map:   0%|          | 0/5 [00:00<?, ? examples/s]

Map:   0%|          | 0/5 [00:00<?, ? examples/s]

Starting fine-tuning for Recipe Generator...


Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Recipe model saved to /content/drive/MyDrive/CSET419_Lab11/component2_recipe_generator

=== TESTING RECIPE GENERATOR ===
Dish: Vegetable Stir-fry | Instructions: Butter two slices of avocados in a bowl. Stir in chopped onions, cilantro, lime juice, and a pinch of salt. Toss in chopped tomatoes, lime juice, and a pinch of salt. Heat olive oil in a


In [20]:
# Component-II: Recipe Instruction Generator

# 1. Prepare Recipe Dataset
recipes_corpus = [
    {"text": "Dish: Scrambled Eggs | Instructions: Whisk eggs in a bowl, add salt and pepper. Heat butter in a pan and cook until soft and fluffy."},
    {"text": "Dish: Pasta Marinara | Instructions: Boil water and cook pasta. In another pan, heat olive oil and garlic, add tomato sauce, then mix with pasta."},
    {"text": "Dish: Grilled Cheese | Instructions: Butter two slices of bread. Place cheese between them and grill on a skillet until golden brown on both sides."},
    {"text": "Dish: Fruit Salad | Instructions: Chop apples, bananas, and grapes. Toss them in a bowl with a splash of lime juice and honey."},
    {"text": "Dish: Guacamole | Instructions: Mash ripe avocados in a bowl. Stir in chopped onions, cilantro, lime juice, and a pinch of salt."}
]

recipe_dataset = Dataset.from_list(recipes_corpus)
tokenized_recipes = recipe_dataset.map(lambda x: tokenizer(x['text'], truncation=True,
    padding='max_length', max_length=128), batched=True)
tokenized_recipes = tokenized_recipes.map(lambda x: {'labels': x['input_ids']}, batched=True)

# 2. Fine-tune for Recipes
recipe_training_args = TrainingArguments(
    output_dir='./gpt2-recipes',
    num_train_epochs=10,
    per_device_train_batch_size=2,
    learning_rate=5e-5,
    save_strategy='no',
    report_to='none',
    fp16=torch.cuda.is_available()
)

recipe_trainer = Trainer(
    model=model,
    args=recipe_training_args,
    train_dataset=tokenized_recipes,
    data_collator=data_collator
)

print("Starting fine-tuning for Recipe Generator...")
recipe_trainer.train()

# 3. Save Recipe Model to Drive
recipe_save_path = '/content/drive/MyDrive/CSET419_Lab11/component2_recipe_generator'
model.save_pretrained(recipe_save_path)
tokenizer.save_pretrained(recipe_save_path)
print(f"\nRecipe model saved to {recipe_save_path}")

# 4. Final Verification
print("\n=== TESTING RECIPE GENERATOR ===")
recipe_prompt = "Dish: Vegetable Stir-fry | Instructions:"
# Using the generate_text function defined in the previous step
print(generate_text(model, tokenizer, recipe_prompt, max_length=60))

Map:   0%|          | 0/5 [00:00<?, ? examples/s]

Map:   0%|          | 0/5 [00:00<?, ? examples/s]

Starting fine-tuning for Recipe Generator...


Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Recipe model saved to /content/drive/MyDrive/CSET419_Lab11/component2_recipe_generator

=== TESTING RECIPE GENERATOR ===
Dish: Vegetable Stir-fry | Instructions: Butter two slices of bread. Place cheese between them and grill on a skillet until golden brown on both sides. Place tomato sauce in a bowl and mix with onions, garlic, chilli, lime juice, and a pinch of salt. Stir


In [16]:
corpus = [
    'this phone has an amazing battery life and the camera quality is outstanding for the price.',
    'i bought this laptop for college and it handles all my assignments and coding projects perfectly.',
    'the sound quality of these headphones is incredible with deep bass and clear vocals.',
    'this smartwatch tracks my steps accurately and the heart rate monitor is very reliable.',
    'great wireless earbuds with noise cancellation that blocks out all background sound.',
    'the keyboard feels very comfortable for long typing sessions and the backlight is a nice touch.',
    'this portable charger saved me during travel and it charges my phone three times on a single charge.',
    'the tablet screen is bright and colorful which makes watching movies a great experience.',
    'i love this fitness tracker because it motivates me to reach my daily exercise goals.',
    'this bluetooth speaker is compact but delivers surprisingly loud and clear audio.',
    'the delivery was fast and the product was packed securely with no damage at all.',
    'excellent value for money and the build quality feels premium despite the affordable price.',
    'the customer service team was very helpful when i had questions about the product features.',
    'this camera takes stunning photos in low light and the video recording quality is very smooth.',
    'i have been using this product for three months and it still works perfectly like day one.',
    'the design is sleek and modern and it looks great on my desk next to my other gadgets.',
    'easy to set up right out of the box and the instructions were clear and simple to follow.',
    'highly recommend this product to anyone looking for quality and reliability at a fair price.',
    'the software updates keep adding new features which makes this purchase even more worthwhile.',
    'best purchase i made this year and i would definitely buy from this brand again.',
]

dataset = Dataset.from_dict({'text': corpus})
tokenized = dataset.map(lambda x: tokenizer(x['text'], truncation=True,
    max_length=128, padding='max_length'), batched=True, remove_columns=['text'])

# Ensure labels are added for loss calculation
tokenized = tokenized.map(lambda x: {'labels': x['input_ids']}, batched=True)

split = tokenized.train_test_split(test_size=0.15, seed=42)

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

training_args = TrainingArguments(
    output_dir='./gpt2-reviews',
    num_train_epochs=15,
    per_device_train_batch_size=4,
    learning_rate=5e-5,
    weight_decay=0.01,
    warmup_steps=50,
    eval_strategy='epoch',
    logging_steps=10,
    save_strategy='no',
    fp16=torch.cuda.is_available(),
    report_to='none'
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=split['train'],
    eval_dataset=split['test'],
    data_collator=data_collator
)

trainer.train()

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss
1,No log,3.348063
2,4.057265,3.247350
3,4.057265,3.106139
4,3.319935,2.981021
5,3.319935,2.881039
6,2.706133,2.803390
7,2.706133,2.756230
8,1.918991,2.740822
9,1.918991,2.765902
10,1.234805,2.867215


TrainOutput(global_step=75, training_loss=1.9250287294387818, metrics={'train_runtime': 9.8613, 'train_samples_per_second': 25.859, 'train_steps_per_second': 7.605, 'total_flos': 16657367040000.0, 'train_loss': 1.9250287294387818, 'epoch': 15.0})

In [17]:
eval_res = trainer.evaluate()
print(f'Perplexity: {math.exp(eval_res["eval_loss"]):.2f}')

print('\n=== FINE-TUNED REVIEWS (After Fine-Tuning) ===')
for p in review_prompts:
    ft_out = generate_text(model, tokenizer, p)
    print(f'Prompt: {p}')
    print(f'  Baseline:   {baseline[p][:120]}...')
    print(f'  Fine-Tuned: {ft_out[:120]}\n')

# Save to Drive as previously requested
final_save_path = '/content/drive/MyDrive/CSET419_Lab11/component1_product_reviews'
model.save_pretrained(final_save_path)
tokenizer.save_pretrained(final_save_path)
print(f'Model saved to {final_save_path}')

Perplexity: 24.74

=== FINE-TUNED REVIEWS (After Fine-Tuning) ===
Prompt: This product is
  Baseline:   This product is made from high quality, lightweight stainless steel. If you are looking for something a little more dura...
  Fine-Tuned: This product is packed with features that make this purchase even more worthwhile. The quality of this product exceeds e

Prompt: I bought this phone and
  Baseline:   I bought this phone and I have not used it on a lot of people. I have also not used it on any other people.

The screen ...
  Fine-Tuned: I bought this phone and it handles all my daily chores perfectly. I would definitely buy from this brand again.

Verifie

Prompt: The quality of this item
  Baseline:   The quality of this item in the item description (and if the item is already in stock) will determine how many times the...
  Fine-Tuned: The quality of this item is exemplary with very little distortion and no audible hum. The build quality is outstanding f



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to /content/drive/MyDrive/CSET419_Lab11/component1_product_reviews


In [13]:
# Sample data for Recipe Instruction Generator
recipes_data = [
    {"text": "Dish: Scrambled Eggs | Instructions: Whisk eggs in a bowl, add salt and pepper. Heat butter in a pan and cook until soft and fluffy."},
    {"text": "Dish: Pasta Marinara | Instructions: Boil water and cook pasta. In another pan, heat olive oil and garlic, add tomato sauce, then mix with pasta."},
    {"text": "Dish: Grilled Cheese | Instructions: Butter two slices of bread. Place cheese between them and grill on a skillet until golden brown on both sides."}
]

# Create recipe dataset
recipe_dataset = Dataset.from_list(recipes_data)
tokenized_recipes = recipe_dataset.map(tokenize_function, batched=True)

# Training setup for Recipe Generator
recipe_output_path = os.path.join(output_path, 'fine_tuned_gpt2_recipes')
training_args_recipe = TrainingArguments(
    output_dir=os.path.join(output_path, 'results_recipe'),
    num_train_epochs=5,
    per_device_train_batch_size=1,
    save_total_limit=1,
    report_to='none'
)

trainer_recipe = Trainer(
    model=model,
    args=training_args_recipe,
    train_dataset=tokenized_recipes,
    data_collator=data_collator,
)

print('Starting fine-tuning for Recipe Generator...')
trainer_recipe.train()

model.save_pretrained(recipe_output_path)
tokenizer.save_pretrained(recipe_output_path)
print(f'Recipe model saved to {recipe_output_path}')

Map:   0%|          | 0/3 [00:00<?, ? examples/s]

Starting fine-tuning for Recipe Generator...


Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Recipe model saved to /content/drive/MyDrive/CSET419_Lab11/fine_tuned_gpt2_recipes


In [12]:
import torch
from transformers import pipeline

# Load the fine-tuned model for inference
fine_tuned_model_path = os.path.join(output_path, 'fine_tuned_gpt2_reviews')

if os.path.exists(fine_tuned_model_path):
    review_gen = pipeline('text-generation', model=fine_tuned_model_path, tokenizer=fine_tuned_model_path)

    prompt = "Product: Wireless Mouse | Review:"
    generated = review_gen(prompt, max_length=50, num_return_sequences=1)

    print("\n--- Generated Review ---")
    print(generated[0]['generated_text'])
else:
    print("Waiting for training to finish and model to be saved...")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Passing `generation_config` together with generation-related arguments=({'max_length', 'num_return_sequences'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=256) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



--- Generated Review ---
Product: Wireless Mouse | Review: Great for gaming, great sound, great battery life, and solid sound. I have two other mice and they sound great.

Verified purchase: This mouse is good for gaming.

Verified purchase: This mouse is great for gaming.

Verified purchase: This mouse is good for gaming.

Verified purchase: This mouse is good for gaming.

Verified purchase: This mouse is good for gaming.

Verified purchase: This mouse is good for gaming.

Verified purchase: Great sound for gaming.

Verified purchase: Great sound for gaming.

Verified purchase: Great sound for gaming.

Verified purchase: Great sound for gaming.

Verified purchase: Great sound for gaming.

Verified purchase: Great sound for gaming.

Verified purchase: Great sound for gaming.

Verified purchase: Great sound for gaming.

Verified purchase: Great sound for gaming.

Verified purchase: Great sound for gaming.

Verified purchase: Great sound for gaming.

Verified purchase: Great sound for g

In [9]:
from transformers import DataCollatorForLanguageModeling

# Prepare data collator for causal LM
tokenizer.pad_token = tokenizer.eos_token
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

training_args = TrainingArguments(
    output_dir=os.path.join(output_path, 'results'),
    overwrite_output_dir=True,
    num_train_epochs=3,
    per_device_train_batch_size=2,
    save_steps=10,
    save_total_limit=2,
    logging_dir=os.path.join(output_path, 'logs'),
    report_to='none'
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_reviews,
    data_collator=data_collator,
)

print('Training setup complete. Starting fine-tuning...')
trainer.train()

# Save the fine-tuned model to Drive
model.save_pretrained(os.path.join(output_path, 'fine_tuned_gpt2_reviews'))
tokenizer.save_pretrained(os.path.join(output_path, 'fine_tuned_gpt2_reviews'))
print(f'Model saved to {output_path}/fine_tuned_gpt2_reviews')

TypeError: TrainingArguments.__init__() got an unexpected keyword argument 'overwrite_output_dir'

In [11]:
from transformers import DataCollatorForLanguageModeling, TrainingArguments, Trainer
import os

# Prepare data collator for causal LM
tokenizer.pad_token = tokenizer.eos_token
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

training_args = TrainingArguments(
    output_dir=os.path.join(output_path, 'results'),
    num_train_epochs=3,
    per_device_train_batch_size=1,
    save_steps=5,
    save_total_limit=1,
    logging_steps=1,
    logging_dir=os.path.join(output_path, 'logs'),
    report_to='none'
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_reviews,
    data_collator=data_collator,
)

print('Starting fine-tuning...')
trainer.train()

# Save the fine-tuned model to Drive
final_path = os.path.join(output_path, 'fine_tuned_gpt2_reviews')
model.save_pretrained(final_path)
tokenizer.save_pretrained(final_path)
print(f'Model and tokenizer saved to {final_path}')

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Starting fine-tuning...


`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
1,4.053099
2,3.292691
3,2.826494
4,3.070062
5,2.114175
6,2.605682
7,1.856696
8,1.549849
9,1.931337
10,1.389423


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model and tokenizer saved to /content/drive/MyDrive/CSET419_Lab11/fine_tuned_gpt2_reviews


In [2]:
!pip install python-docx

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 9.7 MB/s eta 0:00:00


In [5]:
import os
from docx import Document

# Check files in /content to help user debug
print('Files in /content:', os.listdir('/content'))

# Path to the assignment file
doc_path = '/content/CSET419_Lab11_FineTuning.docx'

if os.path.exists(doc_path):
    doc = Document(doc_path)
    # Extracting text to see the instructions
    instructions = [p.text for p in doc.paragraphs if p.text.strip() != '']
    print('\nFile found and loaded. Extracting instructions:')
    for line in instructions[:10]:
        print(line)
else:
    print(f'\nError: File {doc_path} not found. Please upload it to the Files sidebar (left pane) and try again.')

Files in /content: ['.config', 'CSET419_Lab11_FineTuning.docx', 'drive', 'sample_data']

File found and loaded. Extracting instructions:
CSET419 – Introduction to Generative AI
Lab – 11
Objective
The objective of this lab is to fine-tune a pre-trained generative model (GPT-2) for real-world applications. Students will fine-tune GPT-2 to build a Product Review Generator for e-commerce and a Recipe Instruction Generator for a food-tech application, learning how transfer learning adapts a general model to specific business domains.
Learning Outcomes
After completing this lab, students will be able to:
Understand how fine-tuning applies to real-world industry applications
Load and configure a pre-trained GPT-2 model using Hugging Face Transformers
Prepare real-world domain-specific datasets for causal language modeling
Fine-tune the model and compare generated output before and after training
